# 06 permissions

Question: does `golem-permissions` read its configuration as strictly as
`okf/plugins/permissions.md` says, decide every check by the rules of that page in their order, run
the check of the server thread inside its bound and without an allocation, and does the seed
`config/permissions.toml` grant a player of the group default what the group default of LuckPerms on
the live server grants, and nothing of the admin commands, and keep from a player of the group admin
the switches of EssentialsX that LuckPerms keeps from an operator, but the one that the group
default grants, and do the commands of the port of LuckPerms read and change a configuration with
the names, aliases, pages and texts of LuckPerms 5.5.71, each behind its own node, refuse what
Golem does not hold and keep every rule of the format.

`golem-devtool permissions` prints the decision of a configuration for a player and nodes, with
the reason that names the rule, and `golem-devtool permissions-groups` prints the groups of a
player in the order a check searches them. The notebook writes small configurations into a
temporary directory and holds every decision, every reason and every refusal to the expectation
written in its cell. The expectations for the seed come from the wiki of the server,
`okf/modpack/plugins.md` of `skytekx3-1201-server`, section Permissions, which lists the 39 nodes
of the group default. The last two sections time a check and count the allocations of the
check of the server thread.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The seed of the permissions, repository relative.
seed = "config/permissions.toml"

# How often the cost section runs every check, and the bound of one check in nanoseconds. The
# devtool is a debug build, in which a check takes a few microseconds, so the bound only catches
# a check that searches the configuration instead of its tables.
repeat = 20000
max_nanos_per_check = 50000

# The size of the large configuration of the cost section, and how much slower a check in it
# may be than the same check in a configuration with one player.
large_groups = 200
large_players = 5000
max_growth = 10

# The length of a chain of inheritance that loading must take without overflowing its stack.
long_chain = 100000

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo
and rustc. The helpers write a configuration into a temporary directory, run one subcommand on
it and compare a list of cases with what it printed. The players are made up UUIDs.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-06-"))
SEED = P.root / seed

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"

ALEX = "a1e5a1e5-0000-4000-8000-000000000001"
SAM = "5a305a30-0000-4000-8000-000000000002"
NOBODY = "00000000-0000-4000-8000-0000000000ff"
NAMES = {ALEX: "Alex", SAM: "Sam", NOBODY: "nobody"}
COUNTS = {"decisions": 0, "refusals": 0, "group lists": 0}
_written = [0]


def write(text):
    _written[0] += 1
    path = SCRATCH / ("config-%03d.toml" % _written[0])
    path.write_text(text, encoding="utf-8")
    return path


def devtool(*arguments):
    return run([DEVTOOL, *map(str, arguments)], cwd=P.root, timeout=600)


def decide(config, player, nodes, extra=()):
    path = config if isinstance(config, Path) else write(config)
    arguments = ["permissions", "--config", path, "--player", player]
    for node in nodes:
        arguments += ["--node", node]
    result = devtool(*arguments, *extra)
    assert result.returncode == 0, "permissions exited %d\n%s" % (result.returncode, result.stderr)
    return json.loads(result.stdout)


def judge(config, cases):
    # Each case is (player, node, value, reason). The cell prints one row per case and asserts
    # that every decision matches.
    path = config if isinstance(config, Path) else write(config)
    got = {}
    for player in dict.fromkeys(case[0] for case in cases):
        nodes = [case[1] for case in cases if case[0] == player]
        decisions = decide(path, player, nodes)["decisions"]
        assert [d["node"] for d in decisions] == nodes, decisions
        for decision in decisions:
            got[(player, decision["node"])] = decision
    wrong = []
    for player, node, value, reason in cases:
        decision = got[(player, node)]
        ok = decision["value"] is value and decision["reason"] == reason
        COUNTS["decisions"] += 1
        print("%-3s %-7s %-28s %-5s %s" % ("ok" if ok else "BAD", NAMES.get(player, player)[:7], node, decision["value"], decision["reason"]))
        if not ok:
            wrong.append("%s %s: expected %s %r, got %s %r" % (player, node, value, reason, decision["value"], decision["reason"]))
    assert not wrong, "\n".join(wrong)


def groups_of(config, player):
    path = config if isinstance(config, Path) else write(config)
    result = devtool("permissions-groups", "--config", path, "--player", player)
    assert result.returncode == 0, "permissions-groups exited %d\n%s" % (result.returncode, result.stderr)
    COUNTS["group lists"] += 1
    return [(g["group"], g["weight"], g["depth"], g["rank"]) for g in json.loads(result.stdout)["groups"]]


def refusal(text):
    path = write(text)
    result = devtool("permissions", "--config", path, "--player", NOBODY, "--node", "essentials.home")
    prefix = "golem-devtool: permissions: %s: " % path
    assert result.returncode == 1, "the configuration was taken:\n%s\n%s" % (text, result.stdout)
    assert result.stdout == "", "a refused configuration printed on standard output: %r" % result.stdout
    assert result.stderr.startswith(prefix) and result.stderr.endswith("\n"), result.stderr
    return result.stderr[len(prefix):-1]


print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("seed:", P.rel(SEED))

## Exact nodes and wildcards

A node names itself, `prefix.*` names every node below the prefix at any depth but not the prefix
itself, as in LuckPerms, and `*` names every node. An exact deny takes one node out of a wildcard
grant, and the nodes below that node still fall under the wildcard.

In [ ]:
WILDCARDS = '''
[groups.default]
nodes = [
    "essentials.home",
    "essentials.warp.*",
    "-essentials.warp.secret",
    "essentials.signs.use.*",
]
'''
judge(WILDCARDS, [
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
    (NOBODY, "essentials.warp.list", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.warp.list.all", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.warp", None, "No rule names essentials.warp"),
    (NOBODY, "essentials.warp.secret", False, "Group default denies essentials.warp.secret"),
    (NOBODY, "essentials.warp.secret.room", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.signs.use.buy", True, "Group default grants essentials.signs.use.*"),
    (NOBODY, "essentials.signs.use", None, "No rule names essentials.signs.use"),
    (NOBODY, "essentials.fly", None, "No rule names essentials.fly"),
])

## The longest wildcard wins

Within one source the exact node decides first, then the wildcard with the longest prefix, then
`*`. The group below grants everything, denies Essentials, grants the homes again and denies the
secret homes, and every node lands on the most specific of them.

In [ ]:
LONGEST = '''
[groups.default]
nodes = ["*", "-essentials.*", "essentials.home.*", "-essentials.home.secret.*"]
'''
judge(LONGEST, [
    (NOBODY, "minecraft.command.give", True, "Group default grants *"),
    (NOBODY, "fly", True, "Group default grants *"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.*"),
    (NOBODY, "essentials.home", False, "Group default denies essentials.*"),
    (NOBODY, "essentials.home.bed", True, "Group default grants essentials.home.*"),
    (NOBODY, "essentials.home.secret", True, "Group default grants essentials.home.*"),
    (NOBODY, "essentials.home.secret.cave", False, "Group default denies essentials.home.secret.*"),
])

## Specificity over the order of the file

The order in which a source lists its nodes plays no part. Two groups with the same nodes in
opposite orders decide every node the same way.

In [ ]:
FORWARD = '''
[groups.default]
nodes = ["-essentials.*", "essentials.home"]
'''
BACKWARD = '''
[groups.default]
nodes = ["essentials.home", "-essentials.*"]
'''
for config in (FORWARD, BACKWARD):
    judge(config, [
        (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
        (NOBODY, "essentials.fly", False, "Group default denies essentials.*"),
    ])

## A deny wins over a grant at an equal specificity

When one source grants and denies the same node, which is an equal specificity, the deny wins, in
a group as in the nodes of a player, for a node, a wildcard and `*`.

In [ ]:
EQUAL = '''
[groups.default]
nodes = ["essentials.home", "-essentials.home", "essentials.warp.*", "-essentials.warp.*", "*", "-*"]

[players.%s]
nodes = ["essentials.fly", "-essentials.fly"]
''' % ALEX
judge(EQUAL, [
    (NOBODY, "essentials.home", False, "Group default grants and denies essentials.home, and the deny wins"),
    (NOBODY, "essentials.warp.list", False, "Group default grants and denies essentials.warp.*, and the deny wins"),
    (NOBODY, "minecraft.command.give", False, "Group default grants and denies *, and the deny wins"),
    (ALEX, "essentials.fly", False, "The player grants and denies essentials.fly, and the deny wins"),
])

## The player before the groups

The nodes of the player decide first, whatever the groups say. A wildcard deny of the player takes
away a node that a group grants by name, which is where Golem differs from LuckPerms on purpose.
A node the player does not name falls through to the groups.

In [ ]:
PLAYER = '''
[groups.default]
nodes = ["essentials.home", "essentials.suicide", "-essentials.fly"]

[players.%s]
nodes = ["-essentials.suicide", "essentials.fly"]

[players.%s]
nodes = ["-essentials.*"]
''' % (ALEX, SAM)
judge(PLAYER, [
    (ALEX, "essentials.suicide", False, "The player denies essentials.suicide"),
    (ALEX, "essentials.fly", True, "The player grants essentials.fly"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.home", False, "The player denies essentials.*"),
    (SAM, "minecraft.command.give", None, "No rule names minecraft.command.give"),
    (NOBODY, "essentials.suicide", True, "Group default grants essentials.suicide"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.fly"),
])

## The order of weight

The groups are searched in ranks, a higher weight first, and the first rank in which any group
matches decides, even when a group of a lower rank names the node more specifically. Weight comes
before depth, so a group inherited at depth 2 with a higher weight comes before the group that
inherits it.

In [ ]:
WEIGHT = '''
[groups.default]
nodes = ["essentials.home", "-essentials.fly"]

[groups.vip]
weight = 10
nodes = ["essentials.fly", "-essentials.*"]

[groups.staff]
weight = 20
nodes = ["essentials.home"]

[players.%s]
groups = ["vip"]

[players.%s]
groups = ["vip", "staff"]
''' % (ALEX, SAM)
judge(WEIGHT, [
    (ALEX, "essentials.fly", True, "Group vip grants essentials.fly"),
    (ALEX, "essentials.home", False, "Group vip denies essentials.*"),
    (ALEX, "essentials.back", False, "Group vip denies essentials.*"),
    (SAM, "essentials.home", True, "Group staff grants essentials.home"),
    (SAM, "essentials.fly", True, "Group vip grants essentials.fly"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.fly"),
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
])
assert groups_of(WEIGHT, SAM) == [("staff", 20, 1, 0), ("vip", 10, 1, 1), ("default", 0, 1, 2)]
assert groups_of(WEIGHT, NOBODY) == [("default", 0, 1, 0)]

OVER_DEPTH = '''
[groups.default]

[groups.builder]
inherits = ["helper"]
nodes = ["worldedit.wand"]

[groups.helper]
weight = 50
nodes = ["-worldedit.*"]

[players.%s]
groups = ["builder"]
''' % ALEX
judge(OVER_DEPTH, [
    (ALEX, "worldedit.wand", False, "Group helper denies worldedit.*"),
])
assert groups_of(OVER_DEPTH, ALEX) == [("helper", 50, 2, 0), ("builder", 0, 1, 1), ("default", 0, 1, 1)]
print("search orders: as expected")

## The depth of inheritance

At the same weight a closer group comes first. A group the player is in directly has depth 1, the
default group included, and every step of inheritance adds one along the shortest path, so a group
that the player also holds directly moves up to depth 1.

In [ ]:
DEPTH = '''
[groups.default]
nodes = ["essentials.home"]

[groups.member]
inherits = ["base"]
nodes = ["essentials.fly"]

[groups.base]
inherits = ["root"]
nodes = ["-essentials.fly", "essentials.warp"]

[groups.root]
nodes = ["-essentials.warp", "essentials.back"]

[players.%s]
groups = ["member"]

[players.%s]
groups = ["member", "root"]
''' % (ALEX, SAM)
judge(DEPTH, [
    (ALEX, "essentials.fly", True, "Group member grants essentials.fly"),
    (ALEX, "essentials.warp", True, "Group base grants essentials.warp"),
    (ALEX, "essentials.back", True, "Group root grants essentials.back"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.warp", False, "Group root denies essentials.warp"),
    (SAM, "essentials.fly", True, "Group member grants essentials.fly"),
])
assert groups_of(DEPTH, ALEX) == [("default", 0, 1, 0), ("member", 0, 1, 0), ("base", 0, 2, 1), ("root", 0, 3, 2)]
assert groups_of(DEPTH, SAM) == [("default", 0, 1, 0), ("member", 0, 1, 0), ("root", 0, 1, 0), ("base", 0, 2, 1)]
print("search orders: as expected")

## Groups of one rank that disagree

Groups of the same weight and depth do not outrank each other. Each decides on its own, and when
they disagree, the deny wins, also against a grant that names the node more specifically. When
they agree, the first of the rank by name gives the reason.

In [ ]:
RANK = '''
[groups.default]

[groups.blue]
nodes = ["-essentials.fly", "-essentials.warp.*"]

[groups.red]
nodes = ["essentials.fly", "essentials.warp.secret", "essentials.home"]

[groups.green]
nodes = ["essentials.home"]

[players.%s]
groups = ["red", "green", "blue"]
''' % ALEX
judge(RANK, [
    (ALEX, "essentials.fly", False, "Group blue denies essentials.fly and group red of the same rank grants essentials.fly, and the deny wins"),
    (ALEX, "essentials.warp.secret", False, "Group blue denies essentials.warp.* and group red of the same rank grants essentials.warp.secret, and the deny wins"),
    (ALEX, "essentials.warp.list", False, "Group blue denies essentials.warp.*"),
    (ALEX, "essentials.home", True, "Group green grants essentials.home"),
])
assert groups_of(RANK, ALEX) == [("blue", 0, 1, 0), ("default", 0, 1, 0), ("green", 0, 1, 0), ("red", 0, 1, 0)]
print("search order: as expected")

## The default group

Every player is in the default group: a player the configuration does not name, a player with
groups of their own, and a player who names it. What the default group inherits comes at depth 2.

A group without a weight has the weight 0 of the default group and, held directly, the same depth,
so the two share a rank. A wildcard deny of the default group then beats a grant of that group by
name, by the rule of one rank, where LuckPerms grants the node, since it looks up the node itself
in every source before any wildcard. A weight above 0 puts the group first.

The other way round the tie answers as LuckPerms does. A deny of the default group by name beats
a wildcard grant of a group without a weight, and LuckPerms denies the node too, since it finds
the node itself first. A group with a weight above 0 decides before the default group, so its
wildcard grants the node, where LuckPerms denies it.

In [ ]:
DEFAULT = '''
[groups.default]
inherits = ["base"]
nodes = ["essentials.home"]

[groups.base]
nodes = ["essentials.list"]

[groups.vip]
weight = 10
nodes = ["essentials.fly"]

[players.%s]
groups = ["vip"]

[players.%s]
groups = ["default"]
nodes = []
''' % (ALEX, SAM)
judge(DEFAULT, [
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
    (NOBODY, "essentials.list", True, "Group base grants essentials.list"),
    (NOBODY, "essentials.fly", None, "No rule names essentials.fly"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (ALEX, "essentials.fly", True, "Group vip grants essentials.fly"),
    (SAM, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.list", True, "Group base grants essentials.list"),
])
assert groups_of(DEFAULT, NOBODY) == [("default", 0, 1, 0), ("base", 0, 2, 1)]
assert groups_of(DEFAULT, ALEX) == [("vip", 10, 1, 0), ("default", 0, 1, 1), ("base", 0, 2, 2)]
assert groups_of(DEFAULT, SAM) == [("default", 0, 1, 0), ("base", 0, 2, 1)]
print("search orders: as expected")

TIE = '''
[groups.default]
nodes = ["essentials.home", "-worldedit.*"]

[groups.builder]
nodes = ["worldedit.wand"]

[groups.mason]
weight = 1
nodes = ["worldedit.wand"]

[players.%s]
groups = ["builder"]

[players.%s]
groups = ["mason"]
''' % (ALEX, SAM)
judge(TIE, [
    (ALEX, "worldedit.wand", False,
     "Group default denies worldedit.* and group builder of the same rank grants worldedit.wand, and the deny wins"),
    (SAM, "worldedit.wand", True, "Group mason grants worldedit.wand"),
    (SAM, "worldedit.cut", False, "Group default denies worldedit.*"),
])
assert groups_of(TIE, ALEX) == [("builder", 0, 1, 0), ("default", 0, 1, 0)]
assert groups_of(TIE, SAM) == [("mason", 1, 1, 0), ("default", 0, 1, 1)]
print("a group without a weight shares the rank of the default group: as expected")

NAMED = '''
[groups.default]
nodes = ["-worldedit.wand"]

[groups.builder]
nodes = ["worldedit.*"]

[groups.mason]
weight = 1
nodes = ["worldedit.*"]

[players.%s]
groups = ["builder"]

[players.%s]
groups = ["mason"]
''' % (ALEX, SAM)
judge(NAMED, [
    (ALEX, "worldedit.wand", False,
     "Group default denies worldedit.wand and group builder of the same rank grants worldedit.*, and the deny wins"),
    (ALEX, "worldedit.cut", True, "Group builder grants worldedit.*"),
    (SAM, "worldedit.wand", True, "Group mason grants worldedit.*"),
])
assert groups_of(NAMED, ALEX) == [("builder", 0, 1, 0), ("default", 0, 1, 0)]
assert groups_of(NAMED, SAM) == [("mason", 1, 1, 0), ("default", 0, 1, 1)]
print("a deny of the default group by name beats a tied wildcard grant: as expected")

## Unknown nodes and bad input at a check

A node that no rule names is undefined. A check that names no UUID, or a node that is not one
concrete node, is undefined as well, with a reason that says what is wrong. A check reads a UUID in
either case, so the UUID of Alex in upper case finds the nodes of Alex.

In [ ]:
CHECKS = '''
[groups.default]
nodes = ["essentials.home"]

[players.%s]
nodes = ["essentials.fly"]
''' % ALEX
judge(CHECKS, [
    (NOBODY, "golem.unknown", None, "No rule names golem.unknown"),
    ("steve", "essentials.home", None, 'The player "steve" is not a UUID'),
    ("a1e5a1e5000040008000000000000001", "essentials.home", None, 'The player "a1e5a1e5000040008000000000000001" is not a UUID'),
    ("a1e5a1e5-0000-4000-8000-00000000000g", "essentials.home", None, 'The player "a1e5a1e5-0000-4000-8000-00000000000g" is not a UUID'),
    (ALEX.upper(), "essentials.fly", True, "The player grants essentials.fly"),
    (NOBODY, "Essentials.home", None, "The node \"Essentials.home\" holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -"),
    (NOBODY, "essentials.*", None, 'The node "essentials.*" is a wildcard, and a check names one node'),
    (NOBODY, "*", None, 'The node "*" is a wildcard, and a check names one node'),
    (NOBODY, "-essentials.home", None, 'The node "-essentials.home" starts with -, which marks a deny in the configuration'),
    (NOBODY, "", None, 'The node "" is empty'),
    (NOBODY, "essentials..home", None, 'The node "essentials..home" holds an empty segment'),
])

## The answer of LuckPerms on Bukkit

On the live server Arclight asked LuckPerms every Boolean node of the PermissionAPI of Forge for an
online player, and LuckPerms answered a node that a rule names as the rule says and a node that
nothing names false, or true for an operator, after it lowercased the node.
`Permissions::has_permission` keeps that answer, which `golem-devtool permissions` prints for a
player and for an operator beside each decision. A node in upper case finds the rule of its lower
case, where a check of `golem-permissions` refuses it. LuckPerms looks up any text, the text as it
stands and then the wildcards above each of its dots, so a wildcard must also decide a node that
the format refuses: `mod.*` grants a node with a slash, with a colon, with a letter outside ASCII
and the text `mod.*` itself, and `-locked.*` denies a node with a slash also to an operator. The
lower case is the one of Unicode, so the Kelvin sign finds the rule of the letter `k`.

In [ ]:
BUKKIT = '''
[groups.default]
nodes = ["voicechat.speak", "-spark", "mod.*", "-mod.k", "-locked.*"]

[players.%s]
nodes = ["-voicechat.speak"]
''' % ALEX
KELVIN = "mod.\u212a"
NODES = ["voicechat.speak", "spark", "jei.cheat", "VoiceChat.Speak", "-spark", "spark.*",
         "mod.a/b", "mod.a:b", "mod.\u00c9", "mod.*", "locked.a/b", KELVIN]
answers = {}
for player in (ALEX, NOBODY):
    for decision in decide(BUKKIT, player, NODES)["decisions"]:
        answers[(NAMES[player], decision["node"])] = (decision["value"], decision["hasPermission"]["player"],
                                                      decision["hasPermission"]["operator"])
for key, answer in answers.items():
    print("%-7s %-16s check %-5s, as a player %-5s, as an operator %s" % (*key, *answer))
assert answers == {
    ("Alex", "voicechat.speak"): (False, False, False),
    ("Alex", "spark"): (False, False, False),
    ("Alex", "jei.cheat"): (None, False, True),
    ("Alex", "VoiceChat.Speak"): (None, False, False),
    ("Alex", "-spark"): (None, False, True),
    ("Alex", "spark.*"): (None, False, True),
    ("Alex", "mod.a/b"): (None, True, True),
    ("Alex", "mod.a:b"): (None, True, True),
    ("Alex", "mod.\u00c9"): (None, True, True),
    ("Alex", "mod.*"): (None, True, True),
    ("Alex", "locked.a/b"): (None, False, False),
    ("Alex", KELVIN): (None, False, False),
    ("nobody", "voicechat.speak"): (True, True, True),
    ("nobody", "spark"): (False, False, False),
    ("nobody", "jei.cheat"): (None, False, True),
    ("nobody", "VoiceChat.Speak"): (None, True, True),
    ("nobody", "-spark"): (None, False, True),
    ("nobody", "spark.*"): (None, False, True),
    ("nobody", "mod.a/b"): (None, True, True),
    ("nobody", "mod.a:b"): (None, True, True),
    ("nobody", "mod.\u00c9"): (None, True, True),
    ("nobody", "mod.*"): (None, True, True),
    ("nobody", "locked.a/b"): (None, False, False),
    ("nobody", KELVIN): (None, False, False),
}, answers
COUNTS["decisions"] += len(answers)

## Every refusal of the configuration

Loading refuses a configuration that breaks a rule of the format, with exit status 1, nothing on
standard output and one line on standard error that names where. Each case is a configuration and
the exact message after the path of the file.

In [ ]:
U = ALEX
REFUSALS = [
    ("TOML syntax", "[groups.default\n",
     "line 1, column 16: invalid table header, expected `.`, `]`"),
    ("unknown key at the top", "[group.default]\n",
     "line 1, column 2: unknown field `group`, expected `groups` or `players`"),
    ("unknown key of a group", "[groups.default]\nweigth = 1\n",
     "line 2, column 1: unknown field `weigth`, expected one of `weight`, `inherits`, `nodes`"),
    ("unknown key of a player", "[groups.default]\n[players.%s]\ngroup = []\n" % U,
     "line 3, column 1: unknown field `group`, expected `groups` or `nodes`"),
    ("weight not an integer", "[groups.default]\nweight = \"high\"\n",
     "line 2, column 10: invalid type: string \"high\", expected i32"),
    ("weight out of range", "[groups.default]\nweight = 3000000000\n",
     "line 2, column 10: invalid value: integer `3000000000`, expected i32"),
    ("nodes not a list", "[groups.default]\nnodes = \"essentials.home\"\n",
     "line 2, column 9: invalid type: string \"essentials.home\", expected a sequence"),
    ("an empty file", "",
     "groups.default: the group is missing, and every player is in it"),
    ("no default group", "[groups.admin]\nnodes = [\"*\"]\n",
     "groups.default: the group is missing, and every player is in it"),
    ("group name in upper case", "[groups.default]\n[groups.Admin]\n",
     "groups.Admin: the group name \"Admin\" holds the character 'A', but a segment takes only a to z, 0 to 9, _ and -"),
    ("group name with a dot", "[groups.default]\n[groups.\"a.b\"]\n",
     "groups.\"a.b\": the group name \"a.b\" holds a dot, and a group name is one segment"),
]
for name, node, message in [
    ("node in upper case", "Essentials.home", "holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -"),
    ("node with an empty segment", "essentials..home", "holds an empty segment"),
    ("node with a trailing dot", "essentials.", "holds an empty segment"),
    ("node with a leading dot", ".essentials", "holds an empty segment"),
    ("wildcard inside a node", "essentials.*.home", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard glued to a segment", "essentials*", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard at the front", "*.home", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard with no prefix", ".*", "holds an empty segment"),
    ("deny of nothing", "-", "is empty"),
    ("empty node", "", "is empty"),
    ("deny twice", "--essentials.home", "holds the segment -essentials, which starts with -, but a segment starts with a to z, 0 to 9 or _"),
    ("node with a bang", "essentials.home!", "holds the character '!', but a segment takes only a to z, 0 to 9, _ and -"),
    ("node with a space", "essentials. home", "holds the character ' ', but a segment takes only a to z, 0 to 9, _ and -"),
]:
    REFUSALS.append(("group " + name, "[groups.default]\nnodes = [%s]\n" % json.dumps(node),
                     "groups.default.nodes: the node %s %s" % (json.dumps(node), message)))
REFUSALS += [
    ("node named twice", "[groups.default]\nnodes = [\"essentials.home\", \"essentials.home\"]\n",
     "groups.default.nodes: the node \"essentials.home\" is named twice"),
    ("inherits an unknown group", "[groups.default]\n[groups.admin]\ninherits = [\"moderator\"]\n",
     "groups.admin.inherits: the group \"moderator\" is not defined"),
    ("inherits a group twice", "[groups.default]\n[groups.admin]\ninherits = [\"default\", \"default\"]\n",
     "groups.admin.inherits: the group \"default\" is named twice"),
    ("inherits itself", "[groups.default]\n[groups.a]\ninherits = [\"a\"]\n",
     "groups.a.inherits: the group inherits from itself"),
    ("a cycle of two", "[groups.default]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"a\"]\n",
     "groups.b.inherits: the inheritance a, b, a is a cycle"),
    ("a cycle of three", "[groups.default]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"c\"]\n[groups.c]\ninherits = [\"a\"]\n",
     "groups.c.inherits: the inheritance a, b, c, a is a cycle"),
    ("a cycle behind a group", "[groups.default]\ninherits = [\"a\"]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"c\"]\n[groups.c]\ninherits = [\"b\"]\n",
     "groups.c.inherits: the inheritance b, c, b is a cycle"),
    ("a player that is a name", "[groups.default]\n[players.steve]\n",
     "players.steve: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"),
    ("a player in upper case", "[groups.default]\n[players.%s]\n" % U.upper(),
     "players.%s: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001" % U.upper()),
    ("a player without dashes", "[groups.default]\n[players.%s]\n" % U.replace("-", ""),
     "players.%s: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001" % U.replace("-", "")),
    ("a player with the dashes moved", "[groups.default]\n[players.a1e5a1e50-000-4000-8000-000000000001]\n",
     "players.a1e5a1e50-000-4000-8000-000000000001: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"),
    ("a player in an unknown group", "[groups.default]\n[players.%s]\ngroups = [\"vip\"]\n" % U,
     "players.%s.groups: the group \"vip\" is not defined" % U),
    ("a player in a group twice", "[groups.default]\n[groups.vip]\n[players.%s]\ngroups = [\"vip\", \"vip\"]\n" % U,
     "players.%s.groups: the group \"vip\" is named twice" % U),
    ("a player node of bad syntax", "[groups.default]\n[players.%s]\nnodes = [\"Essentials.fly\"]\n" % U,
     "players.%s.nodes: the node \"Essentials.fly\" holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -" % U),
    ("a player node named twice", "[groups.default]\n[players.%s]\nnodes = [\"-essentials.fly\", \"-essentials.fly\"]\n" % U,
     "players.%s.nodes: the node \"-essentials.fly\" is named twice" % U),
]

wrong = []
for name, text, expected in REFUSALS:
    said = refusal(text)
    COUNTS["refusals"] += 1
    ok = said == expected
    print("%-3s %-32s %s" % ("ok" if ok else "BAD", name, said))
    if not ok:
        wrong.append("%s:\n  expected %s\n  got      %s" % (name, expected, said))
assert not wrong, "\n".join(wrong)
print("refusals:", len(REFUSALS))

## A long chain of inheritance

Loading searches the inheritance with a stack of its own, so the default group may inherit through
a chain of `long_chain` groups, a check reaches the end of it, and the depth of the last group is
the length of the chain plus one. The same chain closed into a cycle is refused with every group of
the cycle named.

In [ ]:
lines = ["[groups.default]", "inherits = [\"g000000\"]"]
for index in range(long_chain):
    lines.append("[groups.g%06d]" % index)
    lines.append("inherits = [\"g%06d\"]" % (index + 1) if index + 1 < long_chain else "nodes = [\"essentials.home\"]")
CHAIN = "\n".join(lines) + "\n"
last = "g%06d" % (long_chain - 1)
judge(CHAIN, [(NOBODY, "essentials.home", True, "Group %s grants essentials.home" % last)])
order = groups_of(CHAIN, NOBODY)
assert len(order) == long_chain + 1, len(order)
assert order[0] == ("default", 0, 1, 0) and order[-1] == (last, 0, long_chain + 1, long_chain), (order[0], order[-1])

said = refusal(CHAIN.replace("nodes = [\"essentials.home\"]", "inherits = [\"g000000\"]"))
COUNTS["refusals"] += 1
cycle = ", ".join(["g%06d" % index for index in range(long_chain)] + ["g000000"])
assert said == "groups.%s.inherits: the inheritance %s is a cycle" % (last, cycle), said[:300]
print("groups in the search order:", len(order), "the last at depth", order[-1][2])
print("the cycle:", said[:72], "...", said[-40:])

## A configuration with every kind of key

The counterpart of the refusals: a configuration that uses every key, a deny, both kinds of
wildcard, inheritance over two steps, a player with groups and nodes and a player with neither
loads, and the groups of each player come out in the search order.

In [ ]:
FULL = '''
[groups.default]
weight = 0
inherits = []
nodes = ["essentials.home", "-essentials.home.secret", "essentials.signs.use.*"]

[groups.vip]
weight = 10
inherits = ["member"]
nodes = ["essentials.fly"]

[groups.member]
weight = 5
nodes = ["*", "-golem.*"]

[players.%s]
groups = ["vip"]
nodes = ["-essentials.fly"]

[players.%s]
''' % (ALEX, SAM)
judge(FULL, [
    (ALEX, "essentials.fly", False, "The player denies essentials.fly"),
    (ALEX, "golem.perm", False, "Group member denies golem.*"),
    (ALEX, "essentials.home.secret", True, "Group member grants *"),
    (SAM, "essentials.home.secret", False, "Group default denies essentials.home.secret"),
    (SAM, "essentials.signs.use.buy", True, "Group default grants essentials.signs.use.*"),
])
assert groups_of(FULL, ALEX) == [("vip", 10, 1, 0), ("member", 5, 2, 1), ("default", 0, 1, 2)]
assert groups_of(FULL, SAM) == [("default", 0, 1, 0)]
print("search orders: as expected")

## The seed of SkyTekx

`config/permissions.toml` must load, its group default must hold exactly the 39 nodes that the
wiki of the server lists for the group default of LuckPerms on the Mac mini, and a player that the
store does not name must hold each of them, the signs through their wildcard. The admin commands
stay out of reach of such a player, the nodes of the commands of LuckPerms and of the tools of
Golem among them, and are granted to a player of the group admin, which the cell
adds to a copy of the seed with a made up UUID. The admin holds none of the switches that the
`plugin.yml` of EssentialsX 2.22.0 marks `default: false`, which LuckPerms answers false for an
operator, but `essentials.back.ondeath`, which the group default grants.

In [ ]:
# The group default of the live server, from okf/modpack/plugins.md of skytekx3-1201-server,
# section Permissions, each with the prefix essentials.
WIKI_DEFAULT = [
    "tpa", "tpaccept", "tpdeny", "tpahere", "tpacancel", "tpauto", "home", "home.bed",
    "sethome", "sethome.multiple", "delhome", "back", "back.ondeath", "spawn", "warp",
    "warp.list", "msg", "mail", "mail.send", "ignore", "me", "afk", "list", "seen", "realname",
    "motd", "rules", "help", "helpop", "ping", "getpos", "compass", "depth", "suicide",
    "balance", "balancetop", "pay", "worth", "signs.use.*",
]
EXPECTED = ["essentials." + node for node in WIKI_DEFAULT]
assert len(EXPECTED) == 39 and len(set(EXPECTED)) == 39, "the wiki list holds %d nodes" % len(EXPECTED)

result = devtool("permissions-groups", "--config", SEED, "--player", NOBODY)
assert result.returncode == 0, "the seed does not load: " + result.stderr
seed_groups = json.loads(result.stdout)["groups"]
assert [g["group"] for g in seed_groups] == ["default"], seed_groups
seed_nodes = seed_groups[0]["nodes"]
print("seed default:", len(seed_nodes), "nodes")
assert len(seed_nodes) == len(set(seed_nodes)) == 39, "the seed default holds %d nodes" % len(seed_nodes)
missing, extra = sorted(set(EXPECTED) - set(seed_nodes)), sorted(set(seed_nodes) - set(EXPECTED))
assert not missing and not extra, "missing %s, extra %s" % (missing, extra)

concrete = [node for node in EXPECTED if not node.endswith(".*")]
SIGNS = ["essentials.signs.use.buy", "essentials.signs.use.sell", "essentials.signs.use.warp", "essentials.signs.use.balance"]
judge(SEED, [(NOBODY, node, True, "Group default grants " + node) for node in concrete]
      + [(NOBODY, node, True, "Group default grants essentials.signs.use.*") for node in SIGNS])
print("granted to a default player:", len(concrete), "nodes and", len(SIGNS), "signs")

In [ ]:
ADMIN_ESSENTIALS = [
    "essentials.setspawn", "essentials.setwarp", "essentials.delwarp", "essentials.eco",
    "essentials.gamemode", "essentials.kick", "essentials.ban", "essentials.tpr",
    "essentials.home.others", "essentials.signs.create.buy",
]
ADMIN_GOLEM = ["golem.perm", "golem.player"]
ADMIN_LUCKPERMS = ["luckperms.user.permission.set", "luckperms.creategroup"]
judge(SEED, [(NOBODY, node, None, "No rule names " + node) for node in ADMIN_ESSENTIALS + ADMIN_GOLEM + ADMIN_LUCKPERMS])

seed_with_admin = write(SEED.read_text(encoding="utf-8") + "\n[players.%s]\ngroups = [\"admin\"]\n" % ALEX)
judge(seed_with_admin,
      [(ALEX, node, True, "Group admin grants essentials.*") for node in ADMIN_ESSENTIALS]
      + [(ALEX, node, True, "Group admin grants golem.*") for node in ADMIN_GOLEM]
      + [(ALEX, node, True, "Group admin grants luckperms.*") for node in ADMIN_LUCKPERMS]
      + [(ALEX, node, True, "Group admin grants essentials.*") for node in concrete[:3]]
      + [(NOBODY, "essentials.setspawn", None, "No rule names essentials.setspawn")])
assert groups_of(seed_with_admin, ALEX) == [("admin", 100, 1, 0), ("default", 0, 1, 1)]
print("admin commands refused to a default player and granted to an admin:", len(ADMIN_ESSENTIALS + ADMIN_GOLEM + ADMIN_LUCKPERMS))

# The nodes that the plugin.yml of EssentialsX 2.22.0 marks default: false, which LuckPerms answers
# false for an operator. A deny names one node, so a node below essentials.exempt falls to essentials.*.
DEFAULT_FALSE = [
    "afk.auto", "back.ondeath", "balancetop.exclude", "ban.exempt", "chat.ignoreexempt", "exempt",
    "home.compass", "invsee.preventmodify", "keepinv", "keepxp", "kick.exempt", "kit.exemptdelay",
    "mute.exempt", "near.exclude", "nick.allowunsafe", "nick.hideprefix", "sethome.bed", "silentjoin",
    "silentjoin.vanish", "silentquit", "sudo.exempt", "tempban.exempt", "vanish.pickup",
]
SWITCHES = ["essentials." + node for node in DEFAULT_FALSE if node != "back.ondeath"]
assert len(set(DEFAULT_FALSE)) == 23 and len(SWITCHES) == 22
judge(seed_with_admin,
      [(ALEX, node, False, "Group admin denies " + node) for node in SWITCHES]
      + [(ALEX, "essentials.back.ondeath", True, "Group admin grants essentials.*"),
         (ALEX, "essentials.exempt.protect", True, "Group admin grants essentials.*")]
      + [(NOBODY, node, None, "No rule names " + node) for node in SWITCHES]
      + [(NOBODY, "essentials.back.ondeath", True, "Group default grants essentials.back.ondeath")])
print("switches of EssentialsX that an operator lacks, denied to an admin:", len(SWITCHES))

## The commands of the port of LuckPerms

`golem-devtool permissions-command` runs commands of `/luckperms` on a configuration as the runtime
runs them, one after the other on what the command before left, with a sender that holds a node
of the commands by the rules of the check of a sender, and prints the lines, the changes for the
store, the configuration after a change and the line of a failed save. The console runs them on the
seed with Alex and Sam known, Alex online. The texts are those of LuckPerms 5.5.71 as plain text,
with the context `global`, since Golem has no contexts. A change keeps every rule of the format, so
a parent that closes a cycle of inheritance is refused and changes nothing, and a node that the
format refuses is refused with its reason. The convenience aliases of LuckPerms, such as `u` and
`p`, rewrite their word, and the commands that Golem does not port say so.

In [ ]:
SAM_NAME = {"uuid": SAM, "name": "Sam"}
KNOWN = [{"uuid": ALEX, "name": "Alex", "online": True}, {**SAM_NAME, "online": False}]


def lp(commands, sender=None, config=None, label="lp"):
    script = {"config": config or SEED.read_text(encoding="utf-8"), "sender": sender or {}, "players": KNOWN,
              "commands": [{"label": label, "arguments": arguments} for arguments in commands]}
    path = SCRATCH / ("lp-%03d.json" % _written[0])
    _written[0] += 1
    path.write_text(json.dumps(script), encoding="utf-8")
    result = devtool("permissions-command", path)
    assert result.returncode == 0, "permissions-command exited %d\n%s" % (result.returncode, result.stderr)
    return json.loads(result.stdout)["outcomes"]


CONSOLE_RUNS = [
    "", "help", "user Alex info", "user Alex parent add admin", "user Alex parent info",
    "user Alex permission check essentials.setspawn", "user Alex permission set essentials.fly false",
    "user Alex permission set essentials.fly false", "u Alex p info", "user Alex permission unset essentials.fly",
    "user Alex parent remove default", "creategroup Builder 10", "group builder parent add default",
    "group builder permission set worldedit.*", "group default parent add builder", "user Sam parent add builder",
    "deletegroup builder", "deletegroup default", "deletegroup builder", "listgroups", "group admin info",
    "user Alex permission set Essentials.Fly", "user Alex permission set group.admin",
    "user Alex permission set prefix.10.x", "user Alex parent add admin server=survival", "user Steve info",
    "user bad-name info", "user", "user Alex", "user Alex permission", "user Alex permission set",
    "user Alex meta info", "user Alex permission settemp essentials.fly true 1h", "editor", "nothing",
    "group default listmembers", "user %s info" % SAM.replace("-", ""), "user Alex clear",
]
outcomes = lp(CONSOLE_RUNS)
said = {outcome["arguments"]: outcome for outcome in outcomes}
for outcome in outcomes:
    print("/lp %s" % outcome["arguments"])
    for line in outcome["lines"]:
        print("    " + line)
RUNNING = "[LP] Running LuckPerms v5.5.71, as ported to Golem."
assert said[""]["lines"] == [RUNNING, "[LP] Use /lp help to view available commands."]
assert said["help"]["lines"] == [RUNNING, "> /lp user <user>", "> /lp group <group>", "> /lp creategroup <group>",
                                 "> /lp deletegroup <group>", "> /lp listgroups"]
assert said["user Alex info"]["lines"] == ["[LP] > User Info: Alex", "[LP] - UUID: %s" % ALEX, "[LP]     (type: official)",
                                           "[LP] - Status: Online", "[LP] - Parent Groups:", "[LP]     > default"]
added = said["user Alex parent add admin"]
assert added["lines"] == ["[LP] Alex now inherits permissions from admin in context global."], added
assert added["changes"] == [{"player": ALEX}], added["changes"]
assert '[players.%s]\ngroups = ["admin"]' % ALEX in added["config"], added["config"]
assert added["saveError"] == "[LP] There was an error whilst saving user data for Alex.", added
assert said["user Alex parent info"]["lines"] == ["[LP] Alex's Parents:  (page 1 of 1 - 2 entries)", "> admin",
                                                  "> default"]
assert said["user Alex permission check essentials.setspawn"]["lines"] == [
    "[LP] Permission information for essentials.setspawn:",
    "[LP] - Alex does not have essentials.setspawn set.",
    "[LP] - Alex does not inherit essentials.setspawn.",
    "[LP] - Alex inherits essentials.* set to true from admin in context global.",
    "[LP]",
    "[LP] Permission check for essentials.setspawn:",
    "[LP]     Result: true",
    "[LP]     Processor: golem-permissions",
    "[LP]     Cause: Group admin grants essentials.*",
    "[LP]     Context: global",
]
assert said["user Alex permission set essentials.fly false"]["lines"] == [
    "[LP] Alex already has essentials.fly set in context global."], "the second set is no change"
assert outcomes[6]["lines"] == ["[LP] Set essentials.fly to false for Alex in context global."], outcomes[6]
assert 'nodes = ["-essentials.fly"]' in outcomes[6]["config"], outcomes[6]["config"]
assert said["u Alex p info"]["lines"] == ["[LP] Alex's Permissions:  (page 1 of 1 - 1 entries)", "> -essentials.fly"]
assert said["user Alex permission unset essentials.fly"]["lines"] == [
    "[LP] Unset essentials.fly for Alex in context global."]
assert said["user Alex parent remove default"]["lines"] == [
    "[LP] Every player inherits from default in Golem, so Alex keeps it."]
assert said["creategroup Builder 10"]["lines"] == ["[LP] builder was successfully created."]
assert said["creategroup Builder 10"]["changes"] == [{"group": "builder"}]
assert "[groups.builder]\nweight = 10" in said["creategroup Builder 10"]["config"]
cycle = said["group default parent add builder"]
assert cycle["lines"] == ["[LP] The permissions of Golem refuse the change, so nothing changed: groups.builder.inherits: "
                          "the inheritance default, builder, default is a cycle."], cycle
assert cycle["changes"] == [] and cycle["config"] is None, cycle
deleted = outcomes[16]
assert deleted["lines"] == ["[LP] builder was successfully deleted."], deleted
assert deleted["changes"] == [{"player": SAM}, {"removedGroup": "builder"}], deleted["changes"]
assert "builder" not in deleted["config"], deleted["config"]
assert outcomes[17]["lines"] == ["[LP] You cannot delete the default group."]
assert outcomes[18]["lines"] == ["[LP] An unexpected error occurred. Group not loaded."]
assert said["listgroups"]["lines"] == ["[LP] Showing group entries:    (page 1 of 1 - 2 entries)",
                                       "[LP] Groups: (name, weight, tracks)", "[LP] -  admin - 100",
                                       "[LP] -  default - 0"]
assert said["group admin info"]["lines"] == ["[LP] > Group Info: admin", "[LP] - Display Name: admin",
                                             "[LP] - Weight: 100", "[LP] - Parent Groups:", "[LP]     > default"]
assert said["user Alex permission set Essentials.Fly"]["lines"] == [
    "[LP] Essentials.Fly is no node of the permissions of Golem: it holds the character 'E', but a segment takes "
    "only a to z, 0 to 9, _ and -."]
assert said["user Alex permission set group.admin"]["lines"] == [
    "[LP] Alex already has group.admin set in context global."]
assert said["user Alex permission set prefix.10.x"]["lines"] == [
    "[LP] prefix.10.x is a node of meta in LuckPerms, prefix, which the permissions of Golem do not hold."]
assert said["user Alex parent add admin server=survival"]["lines"] == [
    "[LP] The permissions of Golem have no contexts, so the port takes no server=survival."]
assert said["user Steve info"]["lines"] == ["[LP] A user for Steve could not be found."]
assert said["user bad-name info"]["lines"] == ["[LP] bad-name is not a valid username/uuid."]
assert said["user"]["lines"] == said["user Alex"]["lines"] == [
    "[LP] User Sub Commands: (/lp user <user> ...)", "> info", "> permission", "> parent", "> clear"]
assert said["user Alex permission"]["lines"] == [
    "[LP] Permission Sub Commands: (/lp user <user> permission ...)", "> info - [page]",
    "> set - <node> [true|false]", "> unset - <node>", "> check - <node>", "> clear"]
assert said["user Alex permission set"]["lines"] == [
    "[LP] Command Usage - set", "[LP] > Sets a permission for the object", "[LP] Arguments:",
    "[LP] - <node> -> the permission node to set", "[LP] - [true|false] -> the value of the node"]
NOT_PORTED = "[LP] Golem does not port /lp %s of LuckPerms 5.5.71, see okf/plugins/permissions.md."
assert said["user Alex meta info"]["lines"] == [NOT_PORTED % "user <user> meta"]
assert said["user Alex permission settemp essentials.fly true 1h"]["lines"] == [NOT_PORTED % "user <user> permission settemp"]
assert said["editor"]["lines"] == [NOT_PORTED % "editor"]
assert said["nothing"]["lines"] == said["help"]["lines"]
assert said["group default listmembers"]["lines"] == [
    "[LP] Searching for users and groups who inherit from default...",
    "[LP] Note: when searching for members of the default group, offline players with no other permissions will not be "
    "shown!",
    "[LP] Found 2 entries from 1 users and 1 groups.",
    "[LP] Showing user entries:    (page 1 of 1 - 1 entries)",
    "> Alex",
    "[LP] Showing group entries:    (page 1 of 1 - 1 entries)",
    "> admin"]
assert said["user %s info" % SAM.replace("-", "")]["lines"][:4] == [
    "[LP] > User Info: Sam", "[LP] - UUID: %s" % SAM, "[LP]     (type: official)", "[LP] - Status: Offline"]
assert said["user Alex clear"]["lines"] == ["[LP] Alex's nodes were cleared in context global. (1 node was removed.)"]
assert said["user Alex clear"]["changes"] == [{"player": ALEX}] and ALEX not in said["user Alex clear"]["config"]

## Every command of the port that reads

The cell holds the rest of the commands that change nothing to their lines, on the seed and for the
console. `permission info` shows 19 nodes to a page and `listgroups` 8 groups, a page that the
list does not hold or a word that is no number shows the first, and the groups stand by their
weight and then by their name. The aliases `g`, `t` and `i` as the first word, `perm`, `g`,
`group`, `m`, `i` and `e` as the third and `i` as the fourth rewrite their word before the command
runs, so `t` and `i` alone name the commands `track` and `info` of LuckPerms, which the port does
not take. A subcommand that the port does not know, or that the other kind alone has, such as
`setweight` for a user, answers `Command not recognised.`. A UUID of version 3 is of the type
offline, one of version 2 is an npc, and any other version but 4 is unknown, and a player whom the
server does not know is named Unknown. A main command without its argument answers its usage, and
`creategroup` refuses a name that a group has, a name that is no name of a group and a display
name.

In [ ]:
import tomllib

DEFAULT_NODES = tomllib.loads(SEED.read_text(encoding="utf-8"))["groups"]["default"]["nodes"]
assert len(DEFAULT_NODES) == 39, "the group default of the seed changed, so the pages of this cell differ"
OFFLINE = "0ff11e00-0000-3000-8000-000000000003"
NPC = "0ff11e00-0000-2000-8000-000000000002"
OLD = "0ff11e00-0000-1000-8000-000000000001"


def page(number, first, last):
    return (["[LP] default's Permissions:  (page %d of 3 - 39 entries)" % number]
            + ["> " + node for node in DEFAULT_NODES[first:last]])


def info(name, uuid, kind, status):
    return ["[LP] > User Info: %s" % name, "[LP] - UUID: %s" % uuid, "[LP]     (type: %s)" % kind,
            "[LP] - Status: %s" % status, "[LP] - Parent Groups:", "[LP]     > default"]


ALEX_INFO = info("Alex", ALEX, "official", "Online")
ALEX_PARENTS = ["[LP] Alex's Parents:  (page 1 of 1 - 1 entries)", "> default"]
NO_NODES = ["[LP] Alex does not have any permissions set."]
ADMIN_INFO = said["group admin info"]["lines"]
UNKNOWN = ["[LP] Command not recognised."]
EXISTS = ["[LP] admin already exists!."]
NO_DISPLAY_NAME = ["[LP] The groups of Golem have no display name, so the port takes no Helper."]
READS = [
    ("group default permission info", page(1, 0, 19)),
    ("group default permission info 2", page(2, 19, 38)),
    ("group default permission info 3", page(3, 38, 39)),
    ("group default permission info 4", page(1, 0, 19)),
    ("group default permission info two", page(1, 0, 19)),
    ("g admin info", ADMIN_INFO),
    ("group admin i", ADMIN_INFO),
    ("t", [NOT_PORTED % "track"]),
    ("i", [NOT_PORTED % "info"]),
    ("user Alex i", ALEX_INFO),
    ("user Alex perm info", NO_NODES),
    ("user Alex permission i", NO_NODES),
    ("user Alex g info", ALEX_PARENTS),
    ("user Alex group i", ALEX_PARENTS),
    ("user Alex m", [NOT_PORTED % "user <user> meta"]),
    ("user Alex meta i", [NOT_PORTED % "user <user> meta"]),
    ("user Alex e", [NOT_PORTED % "user <user> editor"]),
    ("user Alex nothing", UNKNOWN),
    ("user Alex setweight 5", UNKNOWN),
    ("user Alex listmembers", UNKNOWN),
    ("user Alex permission nothing", UNKNOWN),
    ("group admin parent nothing", UNKNOWN),
    ("user %s info" % OFFLINE, info("Unknown", OFFLINE, "offline", "Offline")),
    ("user %s info" % NPC, info("Unknown", NPC, "npc", "Offline")),
    ("user %s info" % OLD, info("Unknown", OLD, "unknown", "Offline")),
    ("user %s info" % ALEX.upper(), ALEX_INFO),
    ("creategroup", ["[LP] Command Usage - CreateGroup", "[LP] > Create a new group", "[LP] Arguments:",
                     "[LP] - <name> -> the name of the group", "[LP] - [weight] -> the weight of the group"]),
    ("creategroup admin", EXISTS),
    ("creategroup Admin 5", EXISTS),
    ("creategroup bad.name", ["[LP] bad.name is not a valid group name."]),
    ("creategroup helper Helper", NO_DISPLAY_NAME),
    ("creategroup helper 5 Helper", NO_DISPLAY_NAME),
    ("deletegroup", ["[LP] Command Usage - DeleteGroup", "[LP] > Delete a group", "[LP] Arguments:",
                     "[LP] - <name> -> the name of the group"]),
    ("group nogroup info", ["[LP] A group named nogroup could not be found."]),
    ("group Bad.Name info", ["[LP] A group named bad.name could not be found."]),
    ("group admin setweight", ["[LP] Command Usage - setweight", "[LP] > Set the groups weight", "[LP] Arguments:",
                               "[LP] - <weight> -> the weight to set"]),
    ("group admin setweight heavy", ["[LP] Invalid priority heavy. Expected a number."]),
]
for (command, expected), outcome in zip(READS, lp([command for command, _ in READS]), strict=True):
    assert outcome["lines"] == expected, "/lp %s answered %s, not %s" % (command, outcome["lines"], expected)
    assert outcome["changes"] == [] and outcome["config"] is None and outcome["saveError"] is None, (command, outcome)

# Ten groups make two pages of listgroups. Two of them share a weight, so their names order them.
MADE = ["p1 1", "p2 2", "zeta 3", "alpha 3", "p4 4", "p5 5", "p6 6", "p7 7"]
listed = lp(["creategroup " + group for group in MADE] + ["listgroups", "listgroups 2", "listgroups 3"])[len(MADE):]
ROWS = ["[LP] -  admin - 100", "[LP] -  p7 - 7", "[LP] -  p6 - 6", "[LP] -  p5 - 5", "[LP] -  p4 - 4",
        "[LP] -  alpha - 3", "[LP] -  zeta - 3", "[LP] -  p2 - 2", "[LP] -  p1 - 1", "[LP] -  default - 0"]


def groups_page(number, rows):
    return ["[LP] Showing group entries:    (page %d of 2 - 10 entries)" % number, "[LP] Groups: (name, weight, tracks)",
            *rows]


assert listed[0]["lines"] == groups_page(1, ROWS[:8]), listed[0]["lines"]
assert listed[1]["lines"] == groups_page(2, ROWS[8:]), listed[1]["lines"]
assert listed[2]["lines"] == listed[0]["lines"], listed[2]["lines"]
print("%d commands that read answer their lines, a second and a third page among them" % (len(READS) + 3))

## Every command of the port that changes

The console runs the commands that the cases above leave out, each on what the one before left:
`setweight`, `parent set`, `parent clear`, `permission clear` and `clear`, for a group and for a
player. The cell reads the configuration after each change as TOML. `parent set` leaves the one
group, and for a player the group default, which every player holds, is no entry. A clear counts
what it removes, a clear that finds nothing changes nothing, and a player without a group and
without a node leaves the configuration. A group that is cleared keeps its weight. Every change
names the line of a failed save for its caller, by the kind of its holder.

In [ ]:
GONE = object()
SAVE = {"group": "[LP] There was an error whilst saving group data for builder.",
        "user": "[LP] There was an error whilst saving user data for Alex."}
CLEARED = "[LP] %s's %s were cleared in context global. (%s.)"
ONLY = "[LP] %s had their existing parent groups cleared, and now only inherits %s in context global."
# A command, its lines, and after a change the entry of its holder in the configuration.
CHANGES = [
    ("creategroup builder", ["[LP] builder was successfully created."], {"nodes": []}),
    ("group builder setweight 7", ["[LP] Set weight to 7 for group builder."], {"weight": 7, "nodes": []}),
    ("listgroups", ["[LP] Showing group entries:    (page 1 of 1 - 3 entries)", "[LP] Groups: (name, weight, tracks)",
                    "[LP] -  admin - 100", "[LP] -  builder - 7", "[LP] -  default - 0"], None),
    ("group builder permission set worldedit.wand", ["[LP] Set worldedit.wand to true for builder in context global."],
     {"weight": 7, "nodes": ["worldedit.wand"]}),
    ("group builder permission set worldedit.undo false",
     ["[LP] Set worldedit.undo to false for builder in context global."],
     {"weight": 7, "nodes": ["worldedit.wand", "-worldedit.undo"]}),
    ("group builder parent add default", ["[LP] builder now inherits permissions from default in context global."],
     {"weight": 7, "inherits": ["default"], "nodes": ["worldedit.wand", "-worldedit.undo"]}),
    ("group builder parent set admin", [ONLY % ("builder", "admin")],
     {"weight": 7, "inherits": ["admin"], "nodes": ["worldedit.wand", "-worldedit.undo"]}),
    ("group builder parent info", ["[LP] builder's Parents:  (page 1 of 1 - 1 entries)", "> admin"], None),
    ("group builder permission clear", [CLEARED % ("builder", "permissions", "2 nodes were removed")],
     {"weight": 7, "inherits": ["admin"], "nodes": []}),
    ("group builder permission clear", [CLEARED % ("builder", "permissions", "0 nodes were removed")], None),
    ("group builder parent clear", [CLEARED % ("builder", "parents", "1 node was removed")], {"weight": 7, "nodes": []}),
    ("group builder permission set worldedit.wand", ["[LP] Set worldedit.wand to true for builder in context global."],
     {"weight": 7, "nodes": ["worldedit.wand"]}),
    ("group builder parent add default", ["[LP] builder now inherits permissions from default in context global."],
     {"weight": 7, "inherits": ["default"], "nodes": ["worldedit.wand"]}),
    ("group builder clear", [CLEARED % ("builder", "nodes", "2 nodes were removed")], {"weight": 7, "nodes": []}),
    ("user Alex parent add admin", ["[LP] Alex now inherits permissions from admin in context global."],
     {"groups": ["admin"]}),
    ("user Alex parent add builder", ["[LP] Alex now inherits permissions from builder in context global."],
     {"groups": ["admin", "builder"]}),
    ("user Alex parent set builder", [ONLY % ("Alex", "builder")], {"groups": ["builder"]}),
    ("user Alex parent info", ["[LP] Alex's Parents:  (page 1 of 1 - 2 entries)", "> builder", "> default"], None),
    ("user Alex parent set nogroup", ["[LP] A group named nogroup could not be found."], None),
    ("user Alex parent set admin server=survival",
     ["[LP] The permissions of Golem have no contexts, so the port takes no server=survival."], None),
    ("user Alex permission set essentials.fly", ["[LP] Set essentials.fly to true for Alex in context global."],
     {"groups": ["builder"], "nodes": ["essentials.fly"]}),
    ("user Alex permission set essentials.god false", ["[LP] Set essentials.god to false for Alex in context global."],
     {"groups": ["builder"], "nodes": ["essentials.fly", "-essentials.god"]}),
    ("user Alex permission clear", [CLEARED % ("Alex", "permissions", "2 nodes were removed")], {"groups": ["builder"]}),
    ("user Alex parent add admin", ["[LP] Alex now inherits permissions from admin in context global."],
     {"groups": ["builder", "admin"]}),
    ("user Alex parent clear", [CLEARED % ("Alex", "parents", "2 nodes were removed")], GONE),
    ("user Alex parent clear", [CLEARED % ("Alex", "parents", "0 nodes were removed")], None),
    ("user Alex parent clear global", ["[LP] The permissions of Golem have no contexts, so the port takes no global."],
     None),
    ("user Alex parent add admin", ["[LP] Alex now inherits permissions from admin in context global."],
     {"groups": ["admin"]}),
    ("user Alex permission set essentials.fly", ["[LP] Set essentials.fly to true for Alex in context global."],
     {"groups": ["admin"], "nodes": ["essentials.fly"]}),
    ("user Alex clear", [CLEARED % ("Alex", "nodes", "2 nodes were removed")], GONE),
    ("user Alex parent add builder", ["[LP] Alex now inherits permissions from builder in context global."],
     {"groups": ["builder"]}),
    ("user Alex parent set default", [ONLY % ("Alex", "default")], GONE),
]
for (command, lines, entry), outcome in zip(CHANGES, lp([command for command, _, _ in CHANGES]), strict=True):
    assert outcome["lines"] == lines, "/lp %s answered %s, not %s" % (command, outcome["lines"], lines)
    if entry is None:
        assert outcome["changes"] == [] and outcome["config"] is None and outcome["saveError"] is None, (command, outcome)
        continue
    kind = command.split()[0]
    after = tomllib.loads(outcome["config"])
    if kind == "user":
        assert outcome["changes"] == [{"player": ALEX}] and outcome["saveError"] == SAVE["user"], (command, outcome)
        assert after.get("players", {}).get(ALEX, GONE) == entry, "/lp %s left %s" % (command, after.get("players"))
    else:
        assert outcome["changes"] == [{"group": "builder"}], (command, outcome["changes"])
        assert outcome["saveError"] == ("[LP] There was an error whilst creating builder." if kind == "creategroup"
                                        else SAVE["group"]), (command, outcome["saveError"])
        assert after["groups"]["builder"] == entry, "/lp %s left %s" % (command, after["groups"]["builder"])
print("%d commands that change leave what the page says, and %d of them change nothing" % (
    sum(entry is not None for _, _, entry in CHANGES), sum(entry is None for _, _, entry in CHANGES)))

## Who may run the commands of LuckPerms

A sender may run a command when they hold its node of LuckPerms 5.5.71, such as
`luckperms.user.permission.set`, by the rules of the check of a sender. Sam holds none of them and
is no operator of the game, so the level 4 of the source of their commands gives them nothing: `/lp`
alone tells them so, and any other command answers the version and no command of the list. Once the
console put Sam into the group admin, which grants `luckperms.*`, Sam may run them, and a command
block at level 2 may run them as every source at that level holds every node, while a source below
it may not. Every alias of LuckPerms runs the same command and names itself in the usage.

In [ ]:
AS_SAM = {"uuid": SAM, "name": "Sam", "level": 4}
refused = lp(["", "user Alex info", "listgroups"], sender=AS_SAM)
for outcome in refused:
    print("/lp %-16s %s" % (outcome["arguments"], " / ".join(outcome["lines"])))
assert refused[0]["lines"] == [RUNNING, "[LP] You do not have permission to use any sub commands."], refused[0]
assert refused[1]["lines"] == refused[2]["lines"] == [RUNNING], refused[1:]
with_sam = SEED.read_text(encoding="utf-8") + '\n[players.%s]\ngroups = ["admin"]\n' % SAM
granted = lp(["user Alex info", "listgroups"], sender=AS_SAM, config=with_sam)
assert granted[0]["lines"][0] == "[LP] > User Info: Alex" and granted[1]["lines"][1] == "[LP] Groups: (name, weight, tracks)"
block = lp(["group default permission set essentials.fly"], sender={"name": "@", "level": 2})
assert block[0]["lines"] == ["[LP] Set essentials.fly to true for default in context global."], block
low = lp(["listgroups"], sender={"name": "Server", "level": 1})
assert low[0]["lines"] == [RUNNING], low
for alias in ["luckperms", "perm", "perms", "permission", "permissions"]:
    usage = lp(["help"], label=alias)[0]["lines"]
    assert usage[1] == "> /%s user <user>" % alias, usage
print("Sam is refused, an admin, a command block at level 2 and every alias run the commands")

## The node of every command

Every command of the table of `okf/plugins/permissions.md` asks for its own node. Sam runs each
command once with that node alone, which lets it run, and once with `luckperms.*` and a deny of
that node, which holds every other node of the commands. A subcommand of a user or a group then
answers that Sam may not use it, and a main command answers the usage of the main commands that
Sam still holds, as LuckPerms answers a command that the sender does not hold.

In [ ]:
DENIED = ["[LP] You do not have permission to use this command!"]
TARGETED = ["info", "permission info", "permission set essentials.fly", "permission unset essentials.fly",
            "permission check essentials.fly", "permission clear", "parent info", "parent set default",
            "parent add default", "parent remove default", "parent clear", "clear"]


def node_of(kind, sub):
    words = sub.split()
    return "luckperms.%s.%s" % (kind, ".".join(words[:2] if words[0] in ("permission", "parent") else words[:1]))


NODE_OF = ([("user Alex " + sub, node_of("user", sub)) for sub in TARGETED]
           + [("group admin " + sub, node_of("group", sub)) for sub in TARGETED]
           + [("group admin setweight 50", "luckperms.group.setweight"),
              ("group admin listmembers", "luckperms.group.listmembers"),
              ("creategroup helper", "luckperms.creategroup"), ("deletegroup admin", "luckperms.deletegroup"),
              ("listgroups", "luckperms.listgroups")])
assert len(NODE_OF) == 29 and len({node for _, node in NODE_OF}) == 29, NODE_OF
MAIN_USAGE = {"user": "> /lp user <user>", "group": "> /lp group <group>", "creategroup": "> /lp creategroup <group>",
              "deletegroup": "> /lp deletegroup <group>", "listgroups": "> /lp listgroups"}
for command, node in NODE_OF:
    alone = SEED.read_text(encoding="utf-8") + '\n[players.%s]\nnodes = ["%s"]\n' % (SAM, node)
    without = SEED.read_text(encoding="utf-8") + '\n[players.%s]\nnodes = ["luckperms.*", "-%s"]\n' % (SAM, node)
    held = lp([command], sender=AS_SAM, config=alone)[0]["lines"]
    kept = lp([command], sender=AS_SAM, config=without)[0]["lines"]
    main = command.split()[0]
    assert held and held != DENIED and held[0] != RUNNING, "/lp %s with %s alone answered %s" % (command, node, held)
    if main in ("user", "group"):
        assert kept == DENIED, "/lp %s without %s answered %s" % (command, node, kept)
    else:
        assert kept == [RUNNING] + [usage for name, usage in MAIN_USAGE.items() if name != main], (command, kept)
print("%d commands ask for their own node" % len(NODE_OF))

## The operators of the game and the commands of LuckPerms

LuckPerms has the option `commands-allow-op`, which says whether the operators of the game may use
every command of LuckPerms, and it is on by default. The port keeps that default. A rule of the
permissions decides first: a node that a rule grants a player lets its command run, and a node that
a rule denies them takes its command from an operator too. Where no rule decides, a player whom the
game lists as an operator, at the level 1 or more, runs the command, by the lower of that level and
the level of the source of the command. So a player who is no operator gains nothing at a source of
the level 4, and an operator of the level 4 nothing at a source of the level 0. The sender of a
script names as `operator` the level at which the game lists a player, 0 when left out. The cell
holds Sam to each case on the seed, whose groups name no node of LuckPerms but `luckperms.*` of the
group `admin`: an operator of the lowest level, an operator with a denied node of their own and of
a group, a player who is no operator, and both mixes of the two levels. An operator on a server
without permissions must read the way to the help and not the first steps of LuckPerms, which `/lp`
answers there to a player who may run no command.

The option is no key of the configuration. Its off state is written with a rule, the deny
`-luckperms.*` in the group `default`, which decides for every player, while a group of a higher
weight, such as `admin`, still grants. With it an operator in no such group must be refused as a
player without a node is, a member of `admin` must run the commands, an operator or not, a node of
the player's own must still let its command run and no other, and the console and a command block
must run the commands as before. The console writes the same deny with `lp group default permission
set luckperms.* false`, behind which an operator must be refused as well, and takes it back with
`lp group default permission unset luckperms.*`, behind which an operator must run the commands
again.

In [ ]:
OPERATOR = {"uuid": SAM, "name": "Sam", "level": 1, "operator": 1}
LISTED = "[LP] Showing group entries:    (page 1 of 1 - 2 entries)"
NO_COMMAND = [RUNNING, "[LP] You do not have permission to use any sub commands."]
HELP = [RUNNING, "[LP] Use /lp help to view available commands."]
SEED_TEXT = SEED.read_text(encoding="utf-8")


def with_default(node):
    # The seed with one more node in its group default.
    head = "[groups.default]\nweight = 0\nnodes = [\n"
    assert SEED_TEXT.count(head) == 1, "the seed lost the head of its group default"
    return SEED_TEXT.replace(head, head + '    "%s",\n' % node)


def sam_nodes(*nodes):
    # A table of Sam with nodes of their own, behind a configuration.
    return '\n[players.%s]\nnodes = [%s]\n' % (SAM, ", ".join('"%s"' % node for node in nodes))


# An operator of the lowest level, and no rule: every command runs, one that reads and one that changes.
ran = lp(["", "listgroups", "user Alex info", "creategroup helper"], sender=OPERATOR)
for outcome in ran:
    print("/lp %-44s %s" % (outcome["arguments"], outcome["lines"][0]))
assert ran[0]["lines"] == HELP, "an operator reads no way to the help: %s" % ran[0]
assert ran[1]["lines"][0] == LISTED and ran[2]["lines"][0] == "[LP] > User Info: Alex", ran[1:3]
assert ran[3]["lines"] == ["[LP] helper was successfully created."], ran[3]
assert ran[3]["changes"] == [{"group": "helper"}], ran[3]
assert lp([""], sender=OPERATOR, config="[groups.default]\n")[0]["lines"] == HELP, "an operator reads the first steps"

# A rule decides first: a deny of the player or of a group takes its command from an operator, and no other.
WITHOUT_LISTGROUPS = [RUNNING] + [usage for name, usage in MAIN_USAGE.items() if name != "listgroups"]
for config in (SEED_TEXT + sam_nodes("-luckperms.listgroups"), with_default("-luckperms.listgroups")):
    one_denied = lp(["listgroups", "user Alex info"], sender=OPERATOR, config=config)
    assert one_denied[0]["lines"] == WITHOUT_LISTGROUPS, (
        "a deny did not take the command from an operator: %s" % one_denied[0])
    assert one_denied[1]["lines"][0] == "[LP] > User Info: Alex", one_denied[1]

# No operator, and the lower of the two levels: a player whom the game lists at the level 0, or does not list,
# gains nothing at a source of the level 4, and an operator of the level 4 nothing at a source of the level 0.
for sender in ({"uuid": SAM, "name": "Sam", "level": 4, "operator": 0}, {"uuid": SAM, "name": "Sam", "level": 4},
               {"uuid": SAM, "name": "Sam", "level": 0, "operator": 4}):
    no_operator = lp(["", "listgroups"], sender=sender)
    assert no_operator[0]["lines"] == NO_COMMAND and no_operator[1]["lines"] == [RUNNING], (sender, no_operator)
# An operator of the level 4 at a source of the level 2 is an operator at the level 2.
at_two = lp(["listgroups"], sender={"uuid": SAM, "name": "Sam", "level": 2, "operator": 4})
assert at_two[0]["lines"][0] == LISTED, at_two

# The off state: the group default denies every node of the commands, which decides for every player.
OFF = with_default("-luckperms.*")
turned_off = lp(["", "listgroups", "user Alex info"], sender=OPERATOR, config=OFF)
assert turned_off[0]["lines"] == NO_COMMAND and turned_off[1]["lines"] == turned_off[2]["lines"] == [RUNNING], (
    "the deny of the group default did not take the commands from an operator: %s" % turned_off)
# A member of the group admin, whose weight is higher, runs the commands, an operator or not.
IN_ADMIN = OFF + '\n[players.%s]\ngroups = ["admin"]\n' % SAM
for sender in (OPERATOR, {"uuid": SAM, "name": "Sam", "level": 0}):
    in_admin = lp(["listgroups", "user Alex info"], sender=sender, config=IN_ADMIN)
    assert in_admin[0]["lines"][0] == LISTED and in_admin[1]["lines"][0] == "[LP] > User Info: Alex", (sender, in_admin)
# A node of the player's own still lets its command run, and no other.
own_node = lp(["listgroups", "user Alex info"], sender=OPERATOR, config=OFF + sam_nodes("luckperms.listgroups"))
assert own_node[0]["lines"][0] == LISTED and own_node[1]["lines"] == [RUNNING, MAIN_USAGE["listgroups"]], own_node
# The console and a command block at the level 2 hold every node in either state.
for sender in ({}, {"name": "@", "level": 2}):
    assert lp(["listgroups"], sender=sender, config=OFF)[0]["lines"][0] == LISTED, sender
# The console writes the same deny with a command of the port, and takes it back with another.
by_command = lp(["group default permission set luckperms.* false"])[0]
assert by_command["lines"] == ["[LP] Set luckperms.* to false for default in context global."], by_command
assert '"-luckperms.*"' in by_command["config"], by_command["config"]
assert [outcome["lines"] for outcome in lp(["", "listgroups"], sender=OPERATOR, config=by_command["config"])] == [
    NO_COMMAND, [RUNNING]], "the deny that the console set did not take the commands from an operator"
taken_back = lp(["group default permission unset luckperms.*"], config=by_command["config"])[0]
assert taken_back["lines"] == ["[LP] Unset luckperms.* for default in context global."], taken_back
assert lp(["listgroups"], sender=OPERATOR, config=taken_back["config"])[0]["lines"][0] == LISTED, (
    "an operator does not run the command once the deny is taken back")
print("an operator runs the commands where no rule decides, by the lower of two levels, and the deny of "
      "luckperms.* in the group default takes them from every player whom no higher rule grants them")

## The paths that the cells above leave

The cell holds the rest of the paths of the commands to the texts of LuckPerms 5.5.71, for the
console, each command on what the one before left. A parent is removed, and one that is not held
answers so. A node that is not set answers so, a parent set and unset as the node `group.<name>`
answers with the texts of a node, and Golem refuses a denied parent and a node with a leading
`-`. `permission check` names a node of the holder's own, answers false for a node that the
holder denies and undefined for one that no rule names, and checks a group by the group and its
parents alone, without the group `default` that every player holds. The usage of a group and of
`parent` lists the subcommands with the arguments that the port takes, and a command behind more
words than LuckPerms takes answers its usage under the name that LuckPerms gives it. A group whose
name the format refuses is a group that is not found. An alias in upper case rewrites its word. A
command of LuckPerms that the port does not take says so under the command that holds it in
LuckPerms, so `switchprimarygroup` is one of the parents of a user alone, and `deletegroup` takes a
group out of the group that inherits it, with or without the flag of LuckPerms. Then 17 players of
one group make two pages of `listmembers`, 15 to a page, in the order of `UUID.compareTo` of Java,
which compares the two halves of a UUID as signed numbers. Last, a player who holds no node asks
`/lp` on a server without permissions, which answers the first steps of LuckPerms, and on the seed,
which answers that they may use no command.

In [ ]:
ALEX_CHANGED = [{"player": ALEX}]
SAVE_ALEX = "[LP] There was an error whilst saving user data for Alex."


def check_lines(node, information, result, cause):
    return (["[LP] Permission information for %s:" % node] + ["[LP] - " + line for line in information]
            + ["[LP]", "[LP] Permission check for %s:" % node, "[LP]     Result: %s" % result,
               "[LP]     Processor: golem-permissions", "[LP]     Cause: %s" % cause, "[LP]     Context: global"])


def usage(name, description, *arguments):
    return (["[LP] Command Usage - %s" % name, "[LP] > %s" % description, "[LP] Arguments:"]
            + ["[LP] - %s -> %s" % argument for argument in arguments])


PAGE_ARGUMENT = ("[page]", "the page to view")
NAME_ARGUMENT = ("<name>", "the name of the group")
# Each path: the command, its lines, its changes, and what the configuration after it must hold.
PATHS = [
    ("user Alex parent add admin", ["[LP] Alex now inherits permissions from admin in context global."], ALEX_CHANGED,
     lambda config: config["players"][ALEX] == {"groups": ["admin"]}),
    ("user Alex parent remove admin", ["[LP] Alex no longer inherits permissions from admin in context global."],
     ALEX_CHANGED, lambda config: ALEX not in config.get("players", {})),
    ("user Alex parent remove admin", ["[LP] Alex does not inherit from admin in context global."], [], None),
    ("creategroup builder", ["[LP] builder was successfully created."], [{"group": "builder"}],
     lambda config: config["groups"]["builder"] == {"nodes": []}),
    ("group builder parent add default", ["[LP] builder now inherits permissions from default in context global."],
     [{"group": "builder"}], lambda config: config["groups"]["builder"] == {"inherits": ["default"], "nodes": []}),
    ("group builder parent remove default",
     ["[LP] builder no longer inherits permissions from default in context global."], [{"group": "builder"}],
     lambda config: config["groups"]["builder"] == {"nodes": []}),
    ("group default parent info", ["[LP] default does not have any parents defined."], [], None),
    ("user Alex permission unset essentials.fly", ["[LP] Alex does not have essentials.fly set in context global."], [],
     None),
    ("user Alex permission set group.admin", ["[LP] Set group.admin to true for Alex in context global."], ALEX_CHANGED,
     lambda config: config["players"][ALEX] == {"groups": ["admin"]}),
    ("user Alex permission set group.admin", ["[LP] Alex already has group.admin set in context global."], [], None),
    ("user Alex permission unset group.admin", ["[LP] Unset group.admin for Alex in context global."], ALEX_CHANGED,
     lambda config: ALEX not in config.get("players", {})),
    ("user Alex permission unset group.admin", ["[LP] Alex does not have group.admin set in context global."], [], None),
    ("user Alex permission set group.admin false",
     ["[LP] The permissions of Golem have no denied parent, so Alex keeps its parents. Use /lp user <user> parent "
      "remove admin instead."], [], None),
    ("user Alex permission set -essentials.fly",
     ["[LP] -essentials.fly is no node of the permissions of Golem: it starts with -, which marks a deny, so set "
      "essentials.fly to false instead."], [], None),
    ("user Alex permission set group.nogroup", ["[LP] A group named nogroup could not be found."], [], None),
    ("group builder permission set worldedit.*", ["[LP] Set worldedit.* to true for builder in context global."],
     [{"group": "builder"}], lambda config: config["groups"]["builder"] == {"nodes": ["worldedit.*"]}),
    ("group builder permission unset worldedit.*", ["[LP] Unset worldedit.* for builder in context global."],
     [{"group": "builder"}], lambda config: config["groups"]["builder"] == {"nodes": []}),
    ("group builder permission unset worldedit.*", ["[LP] builder does not have worldedit.* set in context global."],
     [], None),
    ("user Alex permission set essentials.fly false", ["[LP] Set essentials.fly to false for Alex in context global."],
     ALEX_CHANGED, lambda config: config["players"][ALEX] == {"nodes": ["-essentials.fly"]}),
    ("user Alex permission check essentials.fly",
     check_lines("essentials.fly", ["Alex has essentials.fly set to false in context global.",
                                    "Alex does not inherit essentials.fly."], "false",
                 "The player denies essentials.fly"), [], None),
    ("user Alex permission check nothing.names.this",
     check_lines("nothing.names.this", ["Alex does not have nothing.names.this set.",
                                        "Alex does not inherit nothing.names.this."], "undefined",
                 "No rule names nothing.names.this"), [], None),
    ("group admin permission check golem.player",
     check_lines("golem.player", ["admin does not have golem.player set.", "admin does not inherit golem.player.",
                                  "admin has golem.* set to true in context global."], "true",
                 "Group admin grants golem.*"), [], None),
    # The group builder inherits from no group, so the group default, which holds the node, plays no part.
    ("group builder permission check essentials.home",
     check_lines("essentials.home", ["builder does not have essentials.home set.",
                                     "builder does not inherit essentials.home."], "undefined",
                 "No rule names essentials.home"), [], None),
    ("group admin permission check essentials.home",
     check_lines("essentials.home", ["admin does not have essentials.home set.",
                                     "admin inherits essentials.home set to true from default in context global.",
                                     "admin has essentials.* set to true in context global."], "true",
                 "Group admin grants essentials.*"), [], None),
    ("group admin", ["[LP] Group Sub Commands: (/lp group <group> ...)", "> info", "> permission", "> parent",
                     "> listmembers - [page]", "> setweight - <weight>", "> clear"], [], None),
    ("group admin parent", ["[LP] Parent Sub Commands: (/lp group <group> parent ...)", "> info - [page]",
                            "> set - <group>", "> add - <group>", "> remove - <group>", "> clear"], [], None),
    ("group admin setweight 5 6", usage("setweight", "Set the groups weight", ("<weight>", "the weight to set")), [],
     None),
    ("group admin permission info 1 2 3", usage("info", "Lists the permission nodes the object has", PAGE_ARGUMENT),
     [], None),
    ("group admin listmembers 1 2 3",
     usage("listmembers", "Show the users/groups who inherit from this group", PAGE_ARGUMENT), [], None),
    ("listgroups 1 2", usage("ListGroups", "List all groups on the platform", PAGE_ARGUMENT), [], None),
    ("creategroup a 1 b c", usage("CreateGroup", "Create a new group", NAME_ARGUMENT,
                                  ("[weight]", "the weight of the group")), [], None),
    ("deletegroup a b c", usage("DeleteGroup", "Delete a group", NAME_ARGUMENT), [], None),
    ("user Alex parent add Bad.Name", ["[LP] A group named bad.name could not be found."], [], None),
    ("user Alex parent remove bad.name", ["[LP] Alex does not inherit from bad.name in context global."], [], None),
    ("deletegroup bad.name", ["[LP] An unexpected error occurred. Group not loaded."], [], None),
    ("U Alex I", ["[LP] > User Info: Alex", "[LP] - UUID: %s" % ALEX, "[LP]     (type: official)", "[LP] - Status: Online",
                  "[LP] - Parent Groups:", "[LP]     > default"], [], None),
    ("USER Alex PERM INFO", ["[LP] Alex's Permissions:  (page 1 of 1 - 1 entries)", "> -essentials.fly"], [], None),
    ("user Alex parent switchprimarygroup admin", [NOT_PORTED % "user <user> parent switchprimarygroup"], [], None),
    ("group admin parent switchprimarygroup admin", UNKNOWN, [], None),
    ("user Alex promote", [NOT_PORTED % "user <user> promote"], [], None),
    ("group admin promote", UNKNOWN, [], None),
    ("group admin rename x", [NOT_PORTED % "group <group> rename"], [], None),
    ("user Alex rename x", UNKNOWN, [], None),
    ("user Alex permission addtemp x", UNKNOWN, [], None),
    ("user Alex parent settemp x", UNKNOWN, [], None),
    ("creategroup base", ["[LP] base was successfully created."], [{"group": "base"}], None),
    ("creategroup top", ["[LP] top was successfully created."], [{"group": "top"}], None),
    ("group top parent add base", ["[LP] top now inherits permissions from base in context global."],
     [{"group": "top"}], lambda config: config["groups"]["top"] == {"inherits": ["base"], "nodes": []}),
    ("deletegroup base --update-parent-lists", ["[LP] base was successfully deleted."],
     [{"group": "top"}, {"removedGroup": "base"}],
     lambda config: "base" not in config["groups"] and config["groups"]["top"] == {"nodes": []}),
]
outcomes = lp([command for command, _, _, _ in PATHS])
for (command, lines, changes, holds), outcome in zip(PATHS, outcomes, strict=True):
    assert outcome["lines"] == lines, "/lp %s answered %s, not %s" % (command, outcome["lines"], lines)
    assert outcome["changes"] == changes, "/lp %s changed %s, not %s" % (command, outcome["changes"], changes)
    assert (outcome["config"] is None) == (not changes), (command, outcome["config"])
    if holds:
        assert holds(tomllib.loads(outcome["config"])), "/lp %s left %s" % (command, outcome["config"])
assert outcomes[0]["saveError"] == SAVE_ALEX and outcomes[2]["saveError"] is None, outcomes[:3]
assert outcomes[3]["saveError"] == "[LP] There was an error whilst creating builder.", outcomes[3]
assert outcomes[-1]["saveError"] == "[LP] There was an error whilst deleting base.", outcomes[-1]


def java_order(uuid):
    # The place of a UUID in the order of UUID.compareTo of Java: both halves as signed numbers.
    value = int(uuid.replace("-", ""), 16)
    halves = (value >> 64, value & (2 ** 64 - 1))
    return tuple(half - 2 ** 64 if half >= 2 ** 63 else half for half in halves)


FIRSTS = [0x10000000, 0x80000000, 0x7fffffff, 0xffffffff, 0x00000001] + list(range(0x20000000, 0x2000000c))
MEMBERS = ["%08x-0000-4000-8000-%012x" % (first, number) for number, first in enumerate(FIRSTS, 1)]
ORDERED = sorted(MEMBERS, key=java_order)
assert ORDERED != sorted(MEMBERS) and ORDERED[0].startswith("80000000") and ORDERED[-1].startswith("7fffffff"), ORDERED
crowd = SEED.read_text(encoding="utf-8") + "".join('\n[players.%s]\ngroups = ["admin"]\n' % uuid for uuid in MEMBERS)
listed = lp(["group admin listmembers", "group admin listmembers 2", "group admin listmembers 3",
             "group default listmembers"], config=crowd)
SEARCHING = "[LP] Searching for users and groups who inherit from %s..."


def members_page(number, entries):
    return ["[LP] Showing user entries:    (page %d of 2 - 17 entries)" % number] + ["> " + entry for entry in entries]


FOUND = "[LP] Found 17 entries from 17 users and 0 groups."
assert listed[0]["lines"] == [SEARCHING % "admin", FOUND] + members_page(1, ORDERED[:15]), listed[0]["lines"]
assert listed[1]["lines"] == [SEARCHING % "admin", FOUND] + members_page(2, ORDERED[15:]), listed[1]["lines"]
assert listed[2]["lines"] == listed[0]["lines"], listed[2]["lines"]
assert listed[3]["lines"] == [
    SEARCHING % "default",
    "[LP] Note: when searching for members of the default group, offline players with no other permissions will not be "
    "shown!",
    "[LP] Found 18 entries from 17 users and 1 groups.",
    *members_page(1, ORDERED[:15]),
    "[LP] Showing group entries:    (page 1 of 1 - 1 entries)", "> admin"], listed[3]["lines"]

NOBODY = {"uuid": SAM, "name": "Sam", "level": 0}
first_steps = lp([""], sender=NOBODY, config="[groups.default]\n")[0]
assert first_steps["lines"] == [
    RUNNING,
    "[LP] It seems that no permissions have been setup yet!",
    "[LP] Before you can use any of the LuckPerms commands in-game, you need to use the console to give yourself access.",
    "[LP] Open your console and run:",
    "[LP]  > lp user Sam permission set luckperms.* true",
    "[LP] After you've done this, you can begin to define your permission assignments and groups.",
    "[LP] Don't know where to start? Check here: https://luckperms.net/wiki/Usage"], first_steps["lines"]
assert lp([""], sender=NOBODY)[0]["lines"] == [RUNNING, "[LP] You do not have permission to use any sub commands."]
print("%d paths answer the lines of LuckPerms, then two pages of 17 members in the order of Java and the first steps "
      "of a server without permissions" % len(PATHS))

## The cost of a check

A check runs on the server thread, so it must not grow with the configuration. The cell times the
same check of a player in three groups once in a configuration with that player alone and once in
one with `large_groups` groups and `large_players` players, through the form of the check that
builds no reason, and holds both to the bound of the parameters cell.

In [ ]:
import time


def configuration(groups, players):
    lines = ["[groups.default]", "nodes = [\"essentials.home\", \"essentials.signs.use.*\"]"]
    for index in range(groups):
        nodes = ["plugin%03d.node%02d" % (index, n) for n in range(20)] + ["-plugin%03d.secret.*" % index]
        lines += ["", "[groups.g%03d]" % index, "weight = %d" % (index % 7), "nodes = %s" % json.dumps(nodes)]
    for index in range(players):
        uuid = "%08x-0000-4000-8000-%012x" % (index, index)
        member = [index % groups, (index * 7 + 1) % groups, (index * 13 + 2) % groups]
        lines += ["", "[players.%s]" % uuid, "groups = %s" % json.dumps(sorted({"g%03d" % g for g in member}))]
        lines += ["nodes = %s" % json.dumps(["-plugin%03d.node%02d" % (member[0], n) for n in range(3)])]
    return "\n".join(lines) + "\n"


PLAYER0 = "%08x-0000-4000-8000-%012x" % (0, 0)
NODES = ["essentials.home", "essentials.signs.use.buy", "plugin001.node05", "plugin002.secret.x", "plugin000.node01", "essentials.setspawn"]
small = write(configuration(3, 1))
large = write(configuration(large_groups, large_players))
timing = {}
for name, path in (("small", small), ("large", large)):
    started = time.monotonic()
    out = decide(path, PLAYER0, NODES, extra=["--repeat", repeat])
    timing[name] = out["nanos_per_check"]
    print("%-5s %7d bytes, %6.0f ns per check over %d checks, %.2f s for the run" % (
        name, path.stat().st_size, out["nanos_per_check"], repeat * len(NODES), time.monotonic() - started))
    assert [d["value"] for d in out["decisions"]] == [True, True, True, False, False, None], out["decisions"]
growth = timing["large"] / timing["small"]
print("growth: %.2f" % growth)
assert max(timing.values()) < max_nanos_per_check, "a check takes %.0f ns" % max(timing.values())
assert growth < max_growth, "a check in the large configuration is %.1f times slower" % growth

## A check allocates nothing

`Permissions::value`, the form of the check for the server thread, allocates nothing, for a node
that is no node or a player that is no UUID as well, whose reason only `check` builds. The cell
builds a counter of `malloc` and its kin in C with `cc`, which every notebook that builds the
devtool needs already, loads it into the devtool through `LD_PRELOAD` on the C library of GNU, and
runs the same checks once with `--repeat 0` and once with `--repeat` of the parameters cell. Both
runs do the same work but the repeated checks, so they must count the same allocations.

In [ ]:
COUNTER = SCRATCH / "count.c"
COUNTER.write_text(r"""
#include <stddef.h>
#include <stdio.h>
#include <unistd.h>

extern void *__libc_malloc(size_t);
extern void *__libc_calloc(size_t, size_t);
extern void *__libc_realloc(void *, size_t);
extern void *__libc_memalign(size_t, size_t);

static unsigned long allocations;

void *malloc(size_t size) { allocations++; return __libc_malloc(size); }
void *calloc(size_t count, size_t size) { allocations++; return __libc_calloc(count, size); }
void *realloc(void *old, size_t size) { allocations++; return __libc_realloc(old, size); }
void *aligned_alloc(size_t align, size_t size) { allocations++; return __libc_memalign(align, size); }
void *memalign(size_t align, size_t size) { allocations++; return __libc_memalign(align, size); }
int posix_memalign(void **out, size_t align, size_t size)
{
	allocations++;
	*out = __libc_memalign(align, size);
	return *out ? 0 : 12;
}

/* The count goes to standard error when the process exits. */
__attribute__((destructor)) static void report(void)
{
	char line[64];
	int length = snprintf(line, sizeof(line), "allocations: %lu\n", allocations);
	if (length > 0 && write(2, line, (size_t)length) < 0)
		return;
}
""", encoding="utf-8")
COUNTING = SCRATCH / "count.so"
run(["cc", "-shared", "-fPIC", "-O1", "-o", COUNTING, COUNTER], check=True)


def allocations(player, nodes, times):
    # The allocations of one run of the devtool that repeats every check `times` times.
    arguments = ["permissions", "--config", SEED, "--player", player, "--repeat", times]
    for node in nodes:
        arguments += ["--node", node]
    result = run(["env", "LD_PRELOAD=%s" % COUNTING, DEVTOOL, *map(str, arguments)], cwd=P.root, timeout=600)
    assert result.returncode == 0, "permissions exited %d\n%s" % (result.returncode, result.stderr)
    counts = [int(line.split()[-1]) for line in result.stderr.splitlines() if line.startswith("allocations: ")]
    assert len(counts) == 1, "the counter did not report:\n" + result.stderr
    return counts[0]


RUNS = [
    ("nodes of the seed", NOBODY, ["essentials.home", "essentials.signs.use.buy", "essentials.setspawn"]),
    ("nodes that are no nodes", NOBODY, ["Essentials.Home", "-essentials.home", "essentials.*", "*", "",
                                         "essentials..home", "essentials.home!", "-essentials.home.*"]),
    ("a player that is no UUID", "steve", ["essentials.home", "Essentials.Home"]),
]
wrong = []
for name, player, nodes in RUNS:
    base, repeated = allocations(player, nodes, 0), allocations(player, nodes, repeat)
    extra = repeated - base
    print("%-26s %d checks, %d allocations, %d more than without them" % (name, repeat * len(nodes), repeated, extra))
    if extra:
        wrong.append("%s: %d checks allocated %d times" % (name, repeat * len(nodes), extra))
assert not wrong, "\n".join(wrong)

## Verdict

A run that reaches this cell passed every case.

In [ ]:
import shutil

shutil.rmtree(SCRATCH)
print("verdict: %d decisions, %d refusals and %d search orders as expected, the seed grants the %d nodes of the wiki" % (
    COUNTS["decisions"], COUNTS["refusals"], COUNTS["group lists"], len(EXPECTED)))